In [1]:
!pip install pyiqa --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.2/276.2 kB 19.0 MB/s eta 0:00:00


In [2]:
# Install and authorize gdown for folders
#!pip install gdown --upgrade
#!gdown --folder "https://drive.google.com/drive/folders/1eN8Mn1wruhhu98wWJmaQOmYvQfbR0Z20" -O /kaggle/working

#!unzip -q -o /kaggle/working/ST/IC17/test/IC17_test.zip -d /kaggle/working/stt_test
#!unzip -q -o /kaggle/working/ST/IC17/train/IC17_train.zip -d /kaggle/working/stt_train
#!unzip -q /kaggle/working/HT/ht_test.zip -d /kaggle/working/htt_test
#!unzip -q /kaggle/working/HT/ht_train.zip -d /kaggle/working/htt_train

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torch.amp import autocast, GradScaler
from einops import rearrange
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import json
import random
import logging
from PIL import Image
import math

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    """Configuration for MFSSNet training and evaluation"""
    def __init__(self):
        # Model architecture
        self.in_channels = 3
        self.out_channels = 3
        self.dim = 32
        self.num_blocks = [2, 2, 2, 2]
        self.num_heads = [2, 4, 8, 16]
        self.window_sizes = [8, 8, 16, 16]
        self.patch_size = 4
        self.num_prt_layers = 4
        
        # Training
        self.batch_size = 10
        self.epochs = 1
        self.lr = 1e-4
        self.weight_decay = 1e-4
        self.img_size = (64, 256)
        
        # Mixed Precision & Gradient Clipping
        self.use_amp = True  # Automatic Mixed Precision
        self.grad_clip_norm = 1.0  # Gradient clipping max norm
        
        # Advanced Scheduler Configuration
        self.warmup_epochs = 5  # Linear warmup epochs
        self.scheduler_t0 = 50  # CosineAnnealing: epochs until first restart
        self.scheduler_tmult = 2  # CosineAnnealing: period multiplier after restart
        
        # Loss weights
        self.alpha_pixel = 1.0
        self.alpha_perceptual = 0.5
        self.alpha_edge = 0.3
        self.alpha_char = 0.3
        self.alpha_stroke = 0.3
        
        # Evaluation
        self.metrics = ['psnr', 'ssim', 'lpips', 'dists', 'musiq', 'niqe', 'brisque']
        
        # Paths
        self.checkpoint_dir = './runnings'
        self.results_dir = './results/MFSSNet'
        self.analysis_dir = './results/analysis'
        self.manipulated_dir = './results/manipulated'
        
        # Resume training
        self.resume = True
    
    def create_directories(self):
        """Create necessary directories"""
        dirs = [
            self.checkpoint_dir,
            self.results_dir,
            self.analysis_dir,
            f"{self.manipulated_dir}/corrupted",
            f"{self.manipulated_dir}/restored",
            f"{self.manipulated_dir}/clean",
            f"{self.analysis_dir}/comprehensive",
            f"{self.analysis_dir}/lam_maps"
        ]
        for d in dirs:
            Path(d).mkdir(parents=True, exist_ok=True)


# ============================================================================
# CA-SSM (Content-Adaptive State-Space Model)
# ============================================================================

class CA_SSM(nn.Module):
    """
    CA-SSM: Content-Adaptive State-Space Model with Lightweight Selectivity
    
    Novel Contributions:
    1. Content-adaptive gating for input-dependent filtering
    2. Input-dependent output projection C(x) 
    3. Bidirectional processing for text (forward + backward)
    This addresses the limitation of basic SSMs while avoiding Mamba's complexity.
    """
    def __init__(self, dim, d_state=16, d_conv=3, expand=2):
        super().__init__()
        self.dim = dim
        self.d_inner = int(dim * expand)
        self.d_state = d_state
        
        # Input projection
        self.in_proj = nn.Linear(dim, self.d_inner * 2)
        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner, d_conv, 
            padding=d_conv//2, groups=self.d_inner
        )
        
        self.content_gate = nn.Sequential(
            nn.Linear(self.d_inner, self.d_inner // 4),
            nn.GELU(),
            nn.Linear(self.d_inner // 4, self.d_inner),
            nn.Sigmoid()
        )
        
        self.A_log = nn.Parameter(torch.randn(self.d_inner))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        
        # Input-dependent B and dt
        self.x_proj = nn.Linear(self.d_inner, self.d_inner)
        self.dt_proj = nn.Linear(self.d_inner, self.d_inner)
        
        self.c_proj = nn.Linear(self.d_inner, self.d_inner)
        
        self.backward_proj = nn.Linear(self.d_inner, self.d_inner)
        
        self.out_proj = nn.Linear(self.d_inner * 2, dim)  # *2 for bidirectional
        
        # Positional encoding for spatial awareness
        self.pe = nn.Parameter(torch.randn(1, dim, 1, 1) * 0.02)
    
    def _process_direction(self, x, reverse=False):
        """Process sequence in one direction with content-adaptive SSM"""
        if reverse:
            x = torch.flip(x, dims=[1])
        
        # Content-adaptive gating (provides lightweight selectivity)
        gate = self.content_gate(x)  # B, L, D - Input-dependent gate
        
        # SSM computation
        dt = F.softplus(self.dt_proj(x))
        A = -torch.exp(self.A_log)
        B = self.x_proj(x)
        C = self.c_proj(x)  # Input-dependent output projection
        
        # State update with gating
        A_scaled = A.unsqueeze(0).unsqueeze(0) * dt
        state = torch.exp(A_scaled) * x + B
        
        # Apply content-adaptive gate (selective filtering)
        state = state * gate
        
        # Output projection with input-dependent C
        output = state * C + self.D.unsqueeze(0).unsqueeze(0) * x
        
        if reverse:
            output = torch.flip(output, dims=[1])
        
        return output
    
    def forward(self, x):
        B, C, H, W = x.shape
        residual = x
        
        # Add positional encoding
        x = x + self.pe
        
        # Flatten spatial dimensions
        x = rearrange(x, 'b c h w -> b (h w) c')
        
        # Input projection
        x_and_res = self.in_proj(x)
        x, res = x_and_res.split([self.d_inner, self.d_inner], dim=-1)
        
        # Conv1d for local context
        x = rearrange(x, 'b l d -> b d l')
        x = self.conv1d(x)
        x = rearrange(x, 'b d l -> b l d')
        x = F.silu(x)
        
        # Bidirectional processing (forward + backward)
        x_forward = self._process_direction(x, reverse=False)
        x_backward = self._process_direction(x, reverse=True)
        
        # Combine both directions
        x = torch.cat([x_forward, x_backward], dim=-1)
        
        # Gating with residual
        x = x * F.silu(torch.cat([res, res], dim=-1))
        
        # Output projection
        x = self.out_proj(x)
        x = rearrange(x, 'b (h w) c -> b c h w', h=H, w=W)
        
        return x + residual


# ============================================================================
# CG-CSWA (Character-Guided Cyclic Shift Window Attention)
# ============================================================================

class CG_CSWA(nn.Module):
    """
    CG-CSWA: Character-Guided Cyclic Shift Window Attention
    
    Novel Contributions:
    1. Predicts character boundaries to guide attention
    2. Boundary-aware attention bias prevents bleeding across characters
    3. Proper cyclic shifting with attention masking
    4. Text-specific attention mechanism
    
    This addresses the limitation of generic attention that doesn't understand
    character structure, leading to artifacts in degraded text restoration.
    """
    def __init__(self, dim, num_heads=8, window_size=8, shift_size=0):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        
        self.boundary_predictor = nn.Sequential(
            nn.Conv2d(dim, dim // 2, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(dim // 2, dim // 4, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(dim // 4, 1, 1),
            nn.Sigmoid()  # Probability of character boundary
        )
        
        self.boundary_bias_weight = nn.Parameter(torch.zeros(1, num_heads, 1, 1))
        
        # Standard relative position bias
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size - 1) * (2 * window_size - 1), num_heads)
        )
        
        # Get pair-wise relative position index
        coords_h = torch.arange(window_size)
        coords_w = torch.arange(window_size)
        coords = torch.stack(torch.meshgrid([coords_h, coords_w], indexing='ij'))
        coords_flatten = torch.flatten(coords, 1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += window_size - 1
        relative_coords[:, :, 1] += window_size - 1
        relative_coords[:, :, 0] *= 2 * window_size - 1
        relative_position_index = relative_coords.sum(-1)
        self.register_buffer("relative_position_index", relative_position_index)
        
        nn.init.trunc_normal_(self.relative_position_bias_table, std=.02)
        
        # For attention masking in shifted windows
        self.attn_mask = None
    
    def _create_attention_mask(self, H, W):
        """Create attention mask for shifted window attention - PROPER IMPLEMENTATION"""
        if self.shift_size == 0:
            return None
        
        img_mask = torch.zeros((1, H, W, 1))
        h_slices = (slice(0, -self.window_size),
                    slice(-self.window_size, -self.shift_size),
                    slice(-self.shift_size, None))
        w_slices = (slice(0, -self.window_size),
                    slice(-self.window_size, -self.shift_size),
                    slice(-self.shift_size, None))
        cnt = 0
        for h in h_slices:
            for w in w_slices:
                img_mask[:, h, w, :] = cnt
                cnt += 1
        
        # Partition mask into windows
        mask_windows = self.window_partition(img_mask.permute(0, 3, 1, 2), self.window_size)
        mask_windows = mask_windows.view(-1, self.window_size * self.window_size)
        attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
        attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(
            attn_mask == 0, float(0.0)
        )
        
        return attn_mask
    
    def _create_boundary_bias(self, boundary_windows):
        """
        Create attention bias that discourages attention across character boundaries
        boundary_windows: (num_windows, window_size^2, 1)
        """
        # Compute pairwise boundary strength
        # High values when crossing boundaries between positions
        boundary_diff = torch.abs(
            boundary_windows.unsqueeze(2) - boundary_windows.unsqueeze(1)
        )  # (num_windows, window_size^2, window_size^2, 1)
        
        # Strong penalty when crossing high boundaries
        boundary_penalty = -10.0 * boundary_diff.squeeze(-1).unsqueeze(1)  # Add head dimension
        
        return boundary_penalty
    
    def forward(self, x):
        B, C, H, W = x.shape
        r = x
        boundary_map = self.boundary_predictor(x)  # B x 1 x H x W
        
        # Pad if needed
        pad_l = pad_t = 0
        pad_r = (self.window_size - W % self.window_size) % self.window_size
        pad_b = (self.window_size - H % self.window_size) % self.window_size
        x = F.pad(x, (pad_l, pad_r, pad_t, pad_b))
        boundary_map = F.pad(boundary_map, (pad_l, pad_r, pad_t, pad_b))
        _, _, Hp, Wp = x.shape
        
        # Create or update attention mask for shifted windows
        if self.shift_size > 0 and (self.attn_mask is None or self.attn_mask.shape[1] != Hp // self.window_size * Wp // self.window_size):
            self.attn_mask = self._create_attention_mask(Hp, Wp).to(x.device)
        
        # Cyclic shift (if needed)
        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(2, 3))
            shifted_boundary = torch.roll(boundary_map, shifts=(-self.shift_size, -self.shift_size), dims=(2, 3))
        else:
            shifted_x = x
            shifted_boundary = boundary_map
        
        # Partition into windows
        x_windows = self.window_partition(shifted_x, self.window_size)
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)
        
        # Partition boundary map
        boundary_windows = self.window_partition(shifted_boundary, self.window_size)
        boundary_windows = boundary_windows.view(-1, self.window_size * self.window_size, 1)
        
        # Create boundary-aware attention bias
        boundary_attn_bias = self._create_boundary_bias(boundary_windows)
        
        # QKV projection
        qkv = self.qkv(x_windows).reshape(
            -1, self.window_size * self.window_size, 3, self.num_heads, self.head_dim
        )
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        # Attention
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)
        
        # Add relative position bias
        relative_position_bias = self.relative_position_bias_table[
            self.relative_position_index.view(-1)
        ].view(self.window_size * self.window_size, self.window_size * self.window_size, -1)
        relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
        attn = attn + relative_position_bias.unsqueeze(0)
        
        attn = attn + boundary_attn_bias * self.boundary_bias_weight
        
        # Apply attention mask for shifted windows
        if self.attn_mask is not None:
            nW = self.attn_mask.shape[0]
            attn = attn.view(B, nW, self.num_heads, self.window_size * self.window_size, self.window_size * self.window_size)
            attn = attn + self.attn_mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, self.window_size * self.window_size, self.window_size * self.window_size)
        
        attn = attn.softmax(dim=-1)
        
        # Apply attention
        x_windows = (attn @ v).transpose(1, 2).reshape(-1, self.window_size * self.window_size, C)
        x_windows = self.proj(x_windows)
        
        # Merge windows
        x_windows = x_windows.view(-1, self.window_size, self.window_size, C)
        shifted_x = self.window_reverse(x_windows, self.window_size, Hp, Wp)
        
        # Reverse cyclic shift
        if self.shift_size > 0:
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(2, 3))
        else:
            x = shifted_x
        
        # Remove padding
        if pad_r > 0 or pad_b > 0:
            x = x[:, :, :H, :W].contiguous()
        
        return x + r
    
    def window_partition(self, x, window_size):
        """Partition into non-overlapping windows"""
        B, C, H, W = x.shape
        x = x.view(B, C, H // window_size, window_size, W // window_size, window_size)
        windows = x.permute(0, 2, 4, 3, 5, 1).contiguous().view(-1, window_size, window_size, C)
        return windows
    
    def window_reverse(self, windows, window_size, H, W):
        """Reverse window partition"""
        B = int(windows.shape[0] / (H * W / window_size / window_size))
        x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
        x = x.permute(0, 5, 1, 3, 2, 4).contiguous().view(B, -1, H, W)
        return x


# ============================================================================
# GCF: Global Context Fusion
# ============================================================================

class GGCF(nn.Module):
    """Gated Global Context Fusion: Uses a gating mechanism for adaptive context fusion."""
    def __init__(self, dim, hidden_dim=None, dropout=0.1):
        super().__init__()
        hidden_dim = hidden_dim or dim * 2
        
        self.conv1 = nn.Conv2d(dim, hidden_dim, 1)
        self.dwconv = nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1, groups=hidden_dim)
        self.conv2 = nn.Conv2d(hidden_dim, dim, 1)
        self.dropout = nn.Dropout(dropout)
        
        # Global context extraction
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.gmp = nn.AdaptiveMaxPool2d(1)
        self.context_proj = nn.Conv2d(hidden_dim * 2, hidden_dim, 1)
        
        self.fusion_gate = nn.Sequential(
            nn.Conv2d(hidden_dim * 2, hidden_dim, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        residual = x
        x = self.conv1(x)
        x_local = F.gelu(x)
        
        # Extract global context
        gap_feat = self.gap(x_local)
        gmp_feat = self.gmp(x_local)
        global_context = torch.cat([gap_feat, gmp_feat], dim=1)
        global_context = self.context_proj(global_context)
        
        broadcasted_context = global_context.expand_as(x_local)
        gate_input = torch.cat([x_local, broadcasted_context], dim=1)
        fusion_gate = self.fusion_gate(gate_input)
        
        x_fused = x_local * (1 - fusion_gate) + broadcasted_context * fusion_gate
        
        x = self.dwconv(x_fused)
        x = F.gelu(x)
        x = self.conv2(x)
        x = self.dropout(x)
        
        return x + residual


# ============================================================================
# AFM: Adaptive Feature Masking
# ============================================================================

class AFM(nn.Module):
    """AFM: Adaptive Feature Masking - Feature Masking in Mixer"""
    def __init__(self, dim):
        super().__init__()
        self.conv = nn.Conv2d(dim, dim, 3, padding=1)
        self.mask_gen = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(dim, dim, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        feat = self.conv(x)
        mask = self.mask_gen(x)
        return feat * mask + x


# ============================================================================
# FDS: Frequency Decomposition Stage
# ============================================================================

class FDS(nn.Module):
    """FDS: Frequency Decomposition Stage - H-L and L-H frequency processing"""
    def __init__(self, dim):
        super().__init__()
        # High to Low frequency transformation
        self.h2l = nn.Sequential(
            nn.Conv2d(dim, dim//2, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(dim//2, dim, 1)
        )
        
        # Low to High frequency transformation
        self.l2h = nn.Sequential(
            nn.Conv2d(dim, dim//2, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(dim//2, dim, 1)
        )
        
        self.gap = nn.AdaptiveAvgPool2d(1)
        
    def forward(self, x):
        # Extract low frequency (global smooth features)
        x_low = self.gap(x)
        
        # Extract high frequency (detail features)
        x_high = x - F.interpolate(F.avg_pool2d(x, 3, 1, 1), size=x.shape[2:], mode='bilinear')
        
        # Cross-frequency enhancement
        h2l = self.h2l(x_high)
        l2h = F.interpolate(self.l2h(x_low), size=x.shape[2:], mode='bilinear')
        
        return x_high + l2h, x_low.expand_as(x) + h2l


# ============================================================================
# CAM: Channel Attention Module
# ============================================================================

class CAM(nn.Module):
    """CAM: Channel Attention Module for feature recalibration"""
    def __init__(self, dim, reduction=16):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(dim, dim // reduction, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(dim // reduction, dim, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        return self.fc(self.gap(x))


# ============================================================================
# FFM: Frequency Feature Mixer
# ============================================================================

class FFM(nn.Module):
    """FFM: Frequency Feature Mixer - Feature Mixing of Mixer"""
    def __init__(self, dim):
        super().__init__()
        self.spatial_mix = nn.Conv2d(dim, dim, 3, padding=1, groups=dim)
        self.channel_mix = nn.Conv2d(dim, dim, 1)
        self.cam = CAM(dim)
        
    def forward(self, x_h, x_l):
        # Mix high and low frequency features
        x = self.spatial_mix(x_h + x_l)
        x = self.channel_mix(x)
        x = self.cam(x) * x
        return x + x_h + x_l


# ============================================================================
# SRB: Spatial Restoration Block (UPDATED with Novel Components)
# ============================================================================

class SRB(nn.Module):
    """
    SRB: Spatial Restoration Block with Novel Components
    - CA-SSM: Content-Adaptive State-Space Model
    - CG-CSWA: Character-Guided Cyclic Shift Window Attention
    - GGCF: Gated Global Context Fusion
    """
    def __init__(self, dim, num_heads=8, window_size=7, use_shift=False):
        super().__init__()
        
        # NOVEL: Content-Adaptive SSM for long-range dependencies
        self.ca_ssm = CA_SSM(dim)
        
        # NOVEL: Character-Guided Cyclic Shift Window Attention
        shift_size = window_size // 2 if use_shift else 0
        self.cg_cswa = CG_CSWA(dim, num_heads, window_size, shift_size)
        
        # Global Context Fusion
        self.ggcf = GGCF(dim)
        
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.norm3 = nn.LayerNorm(dim)
        
    def forward(self, x):
        # Long-range with CA-SSM
        x = x + self.ca_ssm(self.apply_norm(x, self.norm1))
        
        # Character-guided attention with CG-CSWA
        x = x + self.cg_cswa(self.apply_norm(x, self.norm2))
        
        # Global context fusion
        x = x + self.ggcf(self.apply_norm(x, self.norm3))
        
        return x
    
    def apply_norm(self, x, norm):
        B, C, H, W = x.shape
        x = rearrange(x, 'b c h w -> b h w c')
        x = norm(x)
        x = rearrange(x, 'b h w c -> b c h w')
        return x


# ============================================================================
# FRB: Frequency Restoration Block
# ============================================================================

class FRB(nn.Module):
    """FRB: Frequency Restoration Block - AFM + FDS + FFM"""
    def __init__(self, dim):
        super().__init__()
        self.afm = AFM(dim)
        self.fds = FDS(dim)
        self.ffm = FFM(dim)
        
    def forward(self, x):
        r = x
        # Adaptive feature masking
        x_masked = self.afm(x)
        
        # Frequency decomposition
        x_high, x_low = self.fds(x_masked)
        
        # Frequency feature mixing
        x_freq = self.ffm(x_high, x_low)
        
        return x_freq + r


# ============================================================================
# HRB: Hierarchical Restoration Block
# ============================================================================

class HRB(nn.Module):
    """HRB: Hierarchical Restoration Block - Dual-path processing"""
    def __init__(self, dim, num_heads=8, window_size=7, use_shift=False):
        super().__init__()
        # Spatial restoration path (with novel components)
        self.srb = SRB(dim, num_heads, window_size, use_shift)
        
        # Frequency restoration path
        self.frb = FRB(dim)
        
        # Path fusion
        self.fusion = nn.Conv2d(dim * 2, dim, 1)
        
    def forward(self, x):
        # Spatial restoration: CA-SSM + CG-CSWA + GGCF
        x_spatial = self.srb(x)
        
        # Frequency restoration: AFM + FDS + FFM
        x_freq = self.frb(x)
        
        # Fuse both paths
        x_fused = self.fusion(torch.cat([x_spatial, x_freq], dim=1))
        
        return x_fused + x


# ============================================================================
# PRT: Patch Refinement Transformer
# ============================================================================

class PRTLayer(nn.Module):
    """PRT Layer: Single Patch Refinement Transformer Layer"""
    def __init__(self, dim, num_heads=8, mlp_ratio=4.):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)),
            nn.GELU(),
            nn.Linear(int(dim * mlp_ratio), dim)
        )
        
    def forward(self, x):
        # Self-attention for patch relationships
        x_norm = self.norm1(x)
        x = x + self.attn(x_norm, x_norm, x_norm)[0]
        
        # MLP for patch refinement
        x = x + self.mlp(self.norm2(x))
        
        return x


class PRT(nn.Module):
    """PRT: Patch Refinement Transformer for fine-grained detail enhancement"""
    def __init__(self, dim, num_heads=8, mlp_ratio=4., num_layers=2):
        super().__init__()
        self.layers = nn.ModuleList([
            PRTLayer(dim, num_heads, mlp_ratio) for _ in range(num_layers)
        ])
        
    def forward(self, x):
        r = x
        for layer in self.layers:
            x = layer(x)
        return x + r


# ============================================================================
# MFSSNet: Hierarchical Image Restoration Network (UPDATED)
# ============================================================================

class MFSSNet(nn.Module):
    """
    MFSSNet: A MULTI-LEVEL FLEXIBLE STYLE-INFORMED SEGMENTATION-AWARE PERSPECTIVE
    
    Novel Contributions:
    1. CA-SSM: Content-Adaptive State-Space Model with lightweight selectivity
    2. CG-CSWA: Character-Guided Cyclic Shift Window Attention
    3. HRB: Hierarchical dual-path processing (spatial + frequency)
    4. PRT: Patch Refinement Transformer for final detail enhancement
    """
    def __init__(self, in_channels=3, out_channels=3, dim=64, 
                 num_blocks=[2, 2, 2, 2], 
                 num_heads=[4, 8, 16, 32],
                 window_sizes=[8, 8, 8, 8],
                 patch_size=4,
                 num_layers=4):
        super().__init__()
        
        self.patch_size = patch_size
        
        # SFE: Shallow Feature Extraction
        self.sfe = nn.Conv2d(in_channels, dim, 3, padding=1)
        
        # HFE: Hierarchical Feature Extraction (Encoder)
        self.hfe_blocks = nn.ModuleList()
        self.downsamplers = nn.ModuleList()
        
        current_dim = dim
        for stage_idx, (num, heads, ws) in enumerate(zip(num_blocks[:-1], num_heads[:-1], window_sizes[:-1])):
            blocks = nn.ModuleList([
                HRB(current_dim, heads, ws, use_shift=(j % 2 == 1)) 
                for j in range(num)
            ])
            self.hfe_blocks.append(blocks)
            self.downsamplers.append(nn.Conv2d(current_dim, current_dim*2, 3, stride=2, padding=1))
            current_dim = current_dim * 2
        
        # BNE: Bottleneck Enhancement
        self.bne = nn.ModuleList([
            HRB(current_dim, num_heads[-1], window_sizes[-1], use_shift=(j % 2 == 1))
            for j in range(num_blocks[-1])
        ])
        
        # HFR: Hierarchical Feature Reconstruction (Decoder)
        self.hfr_blocks = nn.ModuleList()
        self.upsamplers = nn.ModuleList()
        self.skip_fusions = nn.ModuleList()
        
        for stage_idx, (num, heads, ws) in enumerate(zip(
            reversed(num_blocks[:-1]), 
            reversed(num_heads[:-1]), 
            reversed(window_sizes[:-1])
        )):
            self.upsamplers.append(nn.ConvTranspose2d(current_dim, current_dim//2, 2, stride=2))
            current_dim = current_dim // 2
            self.skip_fusions.append(nn.Conv2d(current_dim*2, current_dim, 1))
            
            blocks = nn.ModuleList([
                HRB(current_dim, heads, ws, use_shift=(j % 2 == 1))
                for j in range(num)
            ])
            self.hfr_blocks.append(blocks)
        
        # PRT: Patch Refinement Transformer
        patch_dim = patch_size * patch_size * dim
        self.prt = PRT(patch_dim, num_heads=8, mlp_ratio=4., num_layers=num_layers)
        
        # DFR: Deep Feature Reconstruction
        self.dfr = nn.Sequential(
            nn.Conv2d(dim, dim // 2, 7, padding=3),
            nn.GELU(),
            nn.Conv2d(dim // 2, out_channels, 3, padding=1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        # SFE: Shallow Feature Extraction
        shallow_feat = self.sfe(x)
        
        # HFE: Hierarchical Feature Extraction
        feat = shallow_feat
        skips = []
        for blocks, down in zip(self.hfe_blocks, self.downsamplers):
            for block in blocks:
                feat = block(feat)
            skips.append(feat)
            feat = down(feat)
        
        # BNE: Bottleneck Enhancement
        for block in self.bne:
            feat = block(feat)
        
        # HFR: Hierarchical Feature Reconstruction
        for blocks, up, skip_fusion, skip in zip(
            self.hfr_blocks, self.upsamplers, self.skip_fusions, reversed(skips)
        ):
            feat = up(feat)
            feat = skip_fusion(torch.cat([feat, skip], dim=1))
            for block in blocks:
                feat = block(feat)

        patches = feat
        # Reshape to patches
        B, C, H, W = patches.shape
        patch_seq = rearrange(patches, 'b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=self.patch_size, p2=self.patch_size)
        
        # PRT: Patch Refinement Transformer
        refined_seq = self.prt(patch_seq)
        
        # Sequence to patch conversion
        refined_spatial = rearrange(refined_seq, 'b (h w) (p1 p2 c) -> b c (h p1) (w p2)', h=H//self.patch_size, w=W//self.patch_size, 
                                   p1=self.patch_size, p2=self.patch_size)
        
        # DFR: Deep Feature Reconstruction
        output = self.dfr(refined_spatial)
        
        return output


# ============================================================================
# LOSS FUNCTIONS
# ============================================================================

class PerceptualLoss(nn.Module):
    """VGG-based perceptual loss"""
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(pretrained=True)
        self.features = vgg.features[:16].eval()
        for param in self.features.parameters():
            param.requires_grad = False
            
    def forward(self, pred, target):
        pred_feat = self.features(pred)
        target_feat = self.features(target)
        return F.mse_loss(pred_feat, target_feat)

class StrokeConnectivityLoss(nn.Module):
    """Loss to ensure stroke connectivity and character structure"""
    def __init__(self):
        super().__init__()
        
        # Morphological operations kernels
        self.register_buffer('dilate_kernel', torch.ones(1, 1, 3, 3))
        self.register_buffer('erode_kernel', torch.ones(1, 1, 3, 3))
    
    def forward(self, pred, target):
        # Convert to grayscale
        pred_gray = pred.mean(dim=1, keepdim=True)
        target_gray = target.mean(dim=1, keepdim=True)
        
        # Compute skeleton/medial axis approximation
        pred_skeleton = self.approximate_skeleton(pred_gray)
        target_skeleton = self.approximate_skeleton(target_gray)
        
        # Loss on skeleton preservation
        skeleton_loss = F.l1_loss(pred_skeleton, target_skeleton)
        
        # Connectivity loss via morphological operations
        pred_connected = self.morphological_closing(pred_gray)
        target_connected = self.morphological_closing(target_gray)
        connectivity_loss = F.l1_loss(pred_connected, target_connected)
        
        return skeleton_loss + connectivity_loss
    
    def approximate_skeleton(self, x):
        """Approximate skeleton via erosion-based thinning"""
        eroded = F.conv2d(x, self.erode_kernel / 9, padding=1)
        skeleton = torch.clamp(x - eroded, 0, 1)
        return skeleton
    
    def morphological_closing(self, x):
        """Morphological closing: dilation followed by erosion"""
        dilated = F.conv2d(x, self.dilate_kernel / 9, padding=1)
        closed = F.conv2d(dilated, self.erode_kernel / 9, padding=1)
        return closed


class MFSSNetLoss(nn.Module):
    """Comprehensive loss function for MFSSNet model"""
    def __init__(self, alpha_pixel=1.0, alpha_perceptual=0.5, alpha_edge=0.2, alpha_char=0.3, alpha_stroke=0.3):
        super().__init__()
        self.alpha_pixel = alpha_pixel
        self.alpha_perceptual = alpha_perceptual
        self.alpha_edge = alpha_edge
        self.alpha_char = alpha_char
        self.alpha_stroke = alpha_stroke
        
        self.vgg = PerceptualLoss()
        self.stroke_loss = StrokeConnectivityLoss()
        
        self.register_buffer('sobel_x', torch.tensor([[[[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]]]).float())
        self.register_buffer('sobel_y', torch.tensor([[[[-1, -2, -1], [0, 0, 0], [1, 2, 1]]]]).float())
    
    def edge_loss(self, pred, target):
        pred_edges_x = F.conv2d(pred.mean(dim=1, keepdim=True), self.sobel_x, padding=1)
        pred_edges_y = F.conv2d(pred.mean(dim=1, keepdim=True), self.sobel_y, padding=1)
        target_edges_x = F.conv2d(target.mean(dim=1, keepdim=True), self.sobel_x, padding=1)
        target_edges_y = F.conv2d(target.mean(dim=1, keepdim=True), self.sobel_y, padding=1)
        
        return F.l1_loss(pred_edges_x, target_edges_x) + F.l1_loss(pred_edges_y, target_edges_y)

    def character_consistency_loss(self, pred, target):
        patch_size = 16
        unfold = nn.Unfold(kernel_size=patch_size, stride=patch_size // 2)
        
        pred_patches = unfold(pred)
        target_patches = unfold(target)
        
        pred_patches_norm = F.normalize(pred_patches, dim=1)
        target_patches_norm = F.normalize(target_patches, dim=1)
        
        similarity = torch.sum(pred_patches_norm * target_patches_norm, dim=1)
        return 1.0 - similarity.mean()

    def forward(self, pred, target):
        pixel_loss = F.l1_loss(pred, target)
        perceptual_loss = self.vgg(pred, target)
        edge_loss = self.edge_loss(pred, target)
        char_loss = self.character_consistency_loss(pred, target)
        stroke_loss = self.stroke_loss(pred, target)
        
        total_loss = (self.alpha_pixel * pixel_loss + 
                     self.alpha_perceptual * perceptual_loss +
                     self.alpha_edge * edge_loss +
                     self.alpha_char * char_loss +
                     self.alpha_stroke * stroke_loss)
        
        return total_loss, {
            'pixel': pixel_loss.item(),
            'perceptual': perceptual_loss.item(),
            'edge': edge_loss.item(),
            'character': char_loss.item(),
            'stroke': stroke_loss.item()
        }


# ============================================================================
# METRICS
# ============================================================================

class MetricsEvaluator:
    """Centralized metrics evaluation"""
    def __init__(self, metrics=None, device='cuda'):
        self.device = device
        self.metrics_names = metrics if metrics else ['psnr', 'ssim']
        
        # Try to import pyiqa if available
        try:
            import pyiqa
            self.metrics = {}
            for metric_name in self.metrics_names:
                try:
                    self.metrics[metric_name] = pyiqa.create_metric(metric_name, device=device)
                except Exception as e:
                    logger.warning(f"Failed to initialize {metric_name}: {e}")
        except ImportError:
            logger.warning("pyiqa not available, using basic metrics")
            self.metrics = {}
        
        if not self.metrics:
            logger.warning("Using fallback PSNR/SSIM computation")
    
    def compute_psnr(self, pred, target):
        """Compute PSNR"""
        mse = F.mse_loss(pred, target)
        if mse == 0:
            return 100.0
        return 20 * torch.log10(1.0 / torch.sqrt(mse))
    
    def compute_ssim(self, pred, target):
        """Simplified SSIM computation"""
        C1 = 0.01 ** 2
        C2 = 0.03 ** 2
        
        mu_pred = F.avg_pool2d(pred, 3, 1, 1)
        mu_target = F.avg_pool2d(target, 3, 1, 1)
        
        sigma_pred = F.avg_pool2d(pred * pred, 3, 1, 1) - mu_pred * mu_pred
        sigma_target = F.avg_pool2d(target * target, 3, 1, 1) - mu_target * mu_target
        sigma_pred_target = F.avg_pool2d(pred * target, 3, 1, 1) - mu_pred * mu_target
        
        ssim_map = ((2 * mu_pred * mu_target + C1) * (2 * sigma_pred_target + C2)) / \
                   ((mu_pred ** 2 + mu_target ** 2 + C1) * (sigma_pred + sigma_target + C2))
        
        return ssim_map.mean()
    
    def compute(self, pred, target):
        pred = torch.clamp(pred, 0, 1)
        target = torch.clamp(target, 0, 1)
        
        results = {}
        
        if self.metrics:
            for name, metric_fn in self.metrics.items():
                try:
                    if name in ['niqe', 'brisque', 'musiq']:
                        score = metric_fn(pred)
                    else:
                        score = metric_fn(pred, target)
                    results[name] = score.mean().item() if torch.is_tensor(score) else score
                except Exception as e:
                    logger.warning(f"Error computing {name}: {e}")
        else:
            # Fallback to basic metrics
            results['psnr'] = self.compute_psnr(pred, target).item()
            results['ssim'] = self.compute_ssim(pred, target).item()
        
        return results


# ============================================================================
# DATASET
# ============================================================================

class BaseTextDataset(Dataset):
    """Base dataset for text restoration"""
    def __init__(self, img_size=(64, 256), augment=False, config=None):
        self.img_size = img_size
        self.augment = augment
        self.config = config or Config()
        self.image_pairs = []
        
        base_transforms = [transforms.Resize(img_size), transforms.ToTensor()] if img_size else [transforms.ToTensor()]
        self.transform = transforms.Compose(base_transforms)
        self.mask_transform = transforms.Compose([
            transforms.Resize(img_size, interpolation=transforms.InterpolationMode.NEAREST) if img_size else transforms.Lambda(lambda x: x),
            transforms.ToTensor()
        ])
    
    def apply_corruption(self, image):
        """Apply random corruption"""
        noise = torch.randn_like(image) * random.uniform(0.05, 0.15)
        return torch.clamp(image + noise, 0, 1)
    
    def __len__(self):
        return len(self.image_pairs)
    
    def __getitem__(self, idx):
        pair = self.image_pairs[idx]
        
        if 'is_coco' in pair and pair['is_coco']:
            image = pair['image']
            mask = pair.get('mask', torch.ones(1, self.img_size[0], self.img_size[1]))
            corrupted = self.apply_corruption(image.clone())
            
            return {
                'corrupted': torch.clamp(corrupted, 0, 1),
                'clean': torch.clamp(image, 0, 1),
                'mask': torch.clamp(mask, 0, 1),
                'image_id': pair['id'],
                'dataset_type': 'coco'
            }
        else:
            corrupted = Image.open(pair['corrupted']).convert('RGB')
            clean = Image.open(pair['clean']).convert('RGB')
            mask = Image.open(pair['mask']).convert('L')
            
            corrupted = self.transform(corrupted)
            clean = self.transform(clean)
            mask = self.mask_transform(mask)
            
            if self.augment and random.random() < 0.7:
                corrupted = self.apply_corruption(corrupted)
            
            return {
                'corrupted': torch.clamp(corrupted, 0, 1),
                'clean': torch.clamp(clean, 0, 1),
                'mask': torch.clamp(mask, 0, 1),
                'image_id': pair['id'],
                'dataset_type': pair.get('type', 'unknown')
            }


class MergedTextDataset(BaseTextDataset):
    """Merged dataset for synthetic and handwritten text"""
    
    def __init__(self, synthetic_dir=None, handwritten_dir=None, split='train', 
                 augment=False, config=None, img_size=None):
        super().__init__(img_size, augment, config)
        
        if synthetic_dir:
            self._load_synthetic(Path(synthetic_dir), split)
        if handwritten_dir:
            self._load_handwritten(Path(handwritten_dir))
    
    def _load_synthetic(self, root_dir, split):
        if split == 'train':
            incomplete_dir = root_dir / 'train_incomplete'
            complete_dir = root_dir / 'train_f'
            mask_dir = root_dir / 'train_mask'
        else:
            incomplete_dir = root_dir / 't_incomplete'
            complete_dir = root_dir / 't_f'
            mask_dir = root_dir / 't_mask'
        
        if not incomplete_dir.exists():
            logger.warning(f"Synthetic directory not found: {incomplete_dir}")
            return
        
        for img_path in incomplete_dir.glob('*.png'):
            img_id = img_path.stem
            complete_path = complete_dir / f"{img_id}.png"
            mask_path = mask_dir / f"{img_id}.png"
            
            if complete_path.exists() and mask_path.exists():
                self.image_pairs.append({
                    'id': f'syn_{img_id}',
                    'corrupted': str(img_path),
                    'clean': str(complete_path),
                    'mask': str(mask_path),
                    'type': 'synthetic'
                })
    
    def _load_handwritten(self, root_dir):
        corrupted_dir = root_dir / 'Corrupted_Image'
        intact_dir = root_dir / 'Intact_Image'
        mask_dir = root_dir / 'Corrupted_mask'
        
        if not corrupted_dir.exists():
            logger.warning(f"Handwritten directory not found: {corrupted_dir}")
            return
        for img_path in corrupted_dir.glob('*.png'):
            img_id = img_path.stem
            intact_path = intact_dir / f"{img_id}.png"
            mask_path = mask_dir / f"{img_id}.png"
            
            if intact_path.exists() and mask_path.exists():
                self.image_pairs.append({
                    'id': f'hw_{img_id}',
                    'corrupted': str(img_path),
                    'clean': str(intact_path),
                    'mask': str(mask_path),
                    'type': 'handwritten'
                })


# ============================================================================
# TRAINER WITH MIXED PRECISION
# ============================================================================

class MFSSNetTrainer:
    """Trainer for MFSSNet model with Mixed Precision Training"""
    def __init__(self, model, train_loader, val_loader, device, config):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.config = config
        
        self.metrics_evaluator = MetricsEvaluator(
            metrics=config.metrics if hasattr(config, 'metrics') else ['psnr', 'ssim'], 
            device=device
        )
        
        # Optimizer
        self.optimizer = optim.AdamW(
            model.parameters(), 
            lr=config.lr,
            betas=(0.9, 0.999),
            weight_decay=config.weight_decay
        )
        
        # IMPROVED SCHEDULER: Warmup + CosineAnnealingWarmRestarts + ReduceLROnPlateau
        # This combination doesn't require knowing total epochs and provides better training
        self.warmup_epochs = config.warmup_epochs if hasattr(config, 'warmup_epochs') else 5
        self.warmup_steps = self.warmup_epochs * len(train_loader)
        self.current_step = 0
        
        # Main scheduler: Cosine Annealing with Warm Restarts (cycles automatically)
        self.cosine_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            self.optimizer,
            T_0=config.scheduler_t0 if hasattr(config, 'scheduler_t0') else 50,  # First restart at 50 epochs
            T_mult=config.scheduler_tmult if hasattr(config, 'scheduler_tmult') else 2,  # Double period after each restart
            eta_min=config.lr * 0.01  # Minimum LR is 1% of initial LR
        )
        
        # Backup scheduler: Reduce on plateau (adaptive based on validation loss)
        self.plateau_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer,
            mode='min',
            factor=0.5,
            patience=10,
            verbose=True,
            min_lr=config.lr * 0.001
        )
        
        # MIXED PRECISION: Initialize GradScaler
        self.use_amp = config.use_amp if hasattr(config, 'use_amp') else True
        self.scaler = GradScaler(enabled=self.use_amp)
        
        # Gradient clipping norm
        self.grad_clip_norm = config.grad_clip_norm if hasattr(config, 'grad_clip_norm') else 1.0
        
        self.criterion = MFSSNetLoss(
            alpha_pixel=config.alpha_pixel,
            alpha_perceptual=config.alpha_perceptual,
            alpha_edge=config.alpha_edge,
            alpha_char=config.alpha_char,
            alpha_stroke=config.alpha_stroke
        ).to(self.device)
        
        self.current_epoch = 0
        self.best_val_loss = float('inf')
        self.train_losses = defaultdict(list)
        self.val_losses = defaultdict(list)
        self.train_metrics = defaultdict(list)
        self.val_metrics = defaultdict(list)
        
        self.checkpoint_dir = Path(config.checkpoint_dir)
        self.results_dir = Path(config.results_dir)
        
        config.create_directories()
        
        logger.info(f"Mixed Precision Training: {'ENABLED' if self.use_amp else 'DISABLED'}")
        logger.info(f"Gradient Clipping Norm: {self.grad_clip_norm}")
        logger.info(f"Scheduler: Warmup({self.warmup_epochs}E) + CosineAnnealingWarmRestarts + ReduceLROnPlateau")
    
    def get_lr(self):
        """Get current learning rate with warmup"""
        if self.current_step < self.warmup_steps:
            # Linear warmup
            return self.config.lr * (self.current_step / self.warmup_steps)
        else:
            # Use cosine scheduler LR
            return self.optimizer.param_groups[0]['lr']
    
    def update_lr(self):
        """Update learning rate with warmup logic"""
        if self.current_step < self.warmup_steps:
            # Warmup phase: linear increase
            lr = self.config.lr * (self.current_step / self.warmup_steps)
            for param_group in self.optimizer.param_groups:
                param_group['lr'] = lr
        # After warmup, cosine scheduler handles LR updates
    
    def save_checkpoint(self, is_best=False):
        checkpoint = {
            'epoch': self.current_epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'cosine_scheduler_state_dict': self.cosine_scheduler.state_dict(),
            'plateau_scheduler_state_dict': self.plateau_scheduler.state_dict(),
            'scaler_state_dict': self.scaler.state_dict(),  # Save scaler state
            'best_val_loss': self.best_val_loss,
            'current_step': self.current_step,
            'train_losses': dict(self.train_losses),
            'val_losses': dict(self.val_losses),
            'model_architecture': 'MFSSNet (CA-SSM + CG-CSWA)',
            'use_amp': self.use_amp,
            'grad_clip_norm': self.grad_clip_norm,
            'scheduler_type': 'Warmup + CosineAnnealingWarmRestarts + ReduceLROnPlateau'
        }
        
        torch.save(checkpoint, self.checkpoint_dir / 'latest.pth')
        
        if is_best:
            torch.save(checkpoint, self.checkpoint_dir / 'best.pth')
            logger.info(f"Saved best model at epoch {self.current_epoch}")
        
        if self.current_epoch % 25 == 0:
            torch.save(checkpoint, self.checkpoint_dir / f'epoch_{self.current_epoch}.pth')
    
    def load_checkpoint(self, path=None):
        if path is None:
            path = self.checkpoint_dir / 'latest.pth'
        
        if not Path(path).exists():
            logger.info(f"No checkpoint at {path}")
            return
        
        checkpoint = torch.load(path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        
        # Load scheduler states
        if 'cosine_scheduler_state_dict' in checkpoint:
            self.cosine_scheduler.load_state_dict(checkpoint['cosine_scheduler_state_dict'])
        if 'plateau_scheduler_state_dict' in checkpoint:
            self.plateau_scheduler.load_state_dict(checkpoint['plateau_scheduler_state_dict'])
        
        # Load scaler state if available
        if 'scaler_state_dict' in checkpoint:
            self.scaler.load_state_dict(checkpoint['scaler_state_dict'])
        
        self.current_epoch = checkpoint.get('epoch', 0)
        self.current_step = checkpoint.get('current_step', 0)
        self.best_val_loss = checkpoint.get('best_val_loss', float('inf'))
        
        logger.info(f"Resumed MFSSNet training from epoch {self.current_epoch}")
    
    def train_epoch(self):
        self.model.train()
        epoch_losses = defaultdict(float)
        epoch_metrics = defaultdict(list)
        
        pbar = tqdm(self.train_loader, desc=f'MFSSNet Epoch {self.current_epoch+1}', leave=False)
        
        for batch in pbar:
            corrupted = batch['corrupted'].to(self.device)
            clean = batch['clean'].to(self.device)
            
            self.optimizer.zero_grad()
            
            # MIXED PRECISION: Forward pass with autocast
            with autocast('cuda'):
                pred = self.model(corrupted)
                total_loss, loss_dict = self.criterion(pred, clean)
            
            # MIXED PRECISION: Backward pass with scaler
            self.scaler.scale(total_loss).backward()
            
            # GRADIENT CLIPPING: Unscale gradients before clipping
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip_norm)
            
            # MIXED PRECISION: Optimizer step with scaler
            self.scaler.step(self.optimizer)
            self.scaler.update()
            
            # UPDATE LEARNING RATE
            self.current_step += 1
            self.update_lr()
            
            # After warmup, use cosine scheduler per step
            if self.current_step >= self.warmup_steps:
                self.cosine_scheduler.step(self.current_epoch + (self.current_step % len(self.train_loader)) / len(self.train_loader))
            
            # Compute metrics (without autocast for accuracy)
            with torch.no_grad():
                metrics = self.metrics_evaluator.compute(pred, clean)
                for k, v in metrics.items():
                    epoch_metrics[k].append(v)
            
            epoch_losses['total'] += total_loss.item()
            for k, v in loss_dict.items():
                epoch_losses[k] += v
            
            current_lr = self.get_lr()
            pbar.set_postfix({
                'Loss': f"{total_loss.item():.4f}",
                'LR': f"{current_lr:.6f}",
                'Scale': f"{self.scaler.get_scale():.0f}",
                'Phase': 'Warmup' if self.current_step < self.warmup_steps else 'Cosine'
            })
        
        num_batches = len(self.train_loader)
        for k in epoch_losses:
            epoch_losses[k] /= num_batches
            self.train_losses[k].append(epoch_losses[k])
        
        for k, v in epoch_metrics.items():
            if v:
                avg = sum(v) / len(v)
                self.train_metrics[k].append(avg)
        
        return epoch_losses, {k: sum(v)/len(v) for k, v in epoch_metrics.items() if v}
    
    def validate_epoch(self):
        self.model.eval()
        epoch_losses = defaultdict(float)
        epoch_metrics = defaultdict(list)
        
        with torch.no_grad():
            pbar = tqdm(self.val_loader, desc='MFSSNet Validation', leave=False)
            
            for batch in pbar:
                corrupted = batch['corrupted'].to(self.device)
                clean = batch['clean'].to(self.device)
                
                # Use autocast for validation too
                with autocast('cuda'):
                    pred = self.model(corrupted)
                    total_loss, loss_dict = self.criterion(pred, clean)
                
                metrics = self.metrics_evaluator.compute(pred, clean)
                for k, v in metrics.items():
                    epoch_metrics[k].append(v)
                
                epoch_losses['total'] += total_loss.item()
                for k, v in loss_dict.items():
                    epoch_losses[k] += v
        
        num_batches = len(self.val_loader)
        for k in epoch_losses:
            epoch_losses[k] /= num_batches
            self.val_losses[k].append(epoch_losses[k])
        
        for k, v in epoch_metrics.items():
            if v:
                avg = sum(v) / len(v)
                self.val_metrics[k].append(avg)
        
        return epoch_losses, {k: sum(v)/len(v) for k, v in epoch_metrics.items() if v}
    
    def train(self, epochs):
        logger.info(f"Starting MFSSNet training for {epochs} epochs")
        logger.info(f"Model: MFSSNet with {sum(p.numel() for p in self.model.parameters()):,} parameters")
        logger.info(f"Novel Components: CA-SSM + CG-CSWA")
        logger.info(f"Mixed Precision: {'ENABLED' if self.use_amp else 'DISABLED'}")
        logger.info(f"Gradient Clipping: {self.grad_clip_norm}")
        logger.info(f"Scheduler: Warmup({self.warmup_epochs}E) + CosineAnnealingWarmRestarts + ReduceLROnPlateau")
        
        for epoch in range(self.current_epoch, epochs):
            train_losses, train_metrics = self.train_epoch()
            val_losses, val_metrics = self.validate_epoch()
            
            # Update plateau scheduler based on validation loss
            self.plateau_scheduler.step(val_losses['total'])
            
            current_lr = self.get_lr()
            log_msg = (f"Epoch {epoch+1}/{epochs} - "
                      f"LR: {current_lr:.6f} - "
                      f"Train Loss: {train_losses['total']:.4f} - "
                      f"Val Loss: {val_losses['total']:.4f}")
            
            for k in ['psnr', 'ssim']:
                if k in train_metrics and k in val_metrics:
                    log_msg += f" - Train {k.upper()}: {train_metrics[k]:.3f}"
                    log_msg += f" - Val {k.upper()}: {val_metrics[k]:.3f}"
            
            logger.info(log_msg)
            
            is_best = val_losses['total'] < self.best_val_loss
            if is_best:
                self.best_val_loss = val_losses['total']
                logger.info(f"🎯 New best validation loss: {self.best_val_loss:.4f}")
            
            self.current_epoch = epoch + 1
            self.save_checkpoint(is_best)


# ============================================================================
# INFERENCER
# ============================================================================

class MFSSNetInferencer:
    """Inferencer for MFSSNet model with LAM visualization and multiple analysis tools"""
    def __init__(self, model, device, config, checkpoint_path=None):
        self.device = device
        self.model = model.to(device)
        self.config = config
        self.metrics_evaluator = MetricsEvaluator(
            metrics=config.metrics if hasattr(config, 'metrics') else ['psnr', 'ssim'], 
            device=device
        )
        
        if checkpoint_path:
            self.load_checkpoint(checkpoint_path)
        
        self.model.eval()
    
    def load_checkpoint(self, path):
        checkpoint = torch.load(path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        logger.info(f"Loaded MFSSNet model from {path}")
    
    def compute_lam(self, corrupted, restored, clean):
        """Compute Local Attribution Map """
        corrupted_input = corrupted.clone().detach().requires_grad_(True)
        
        # We need to forward pass again with grad enabled
        output = self.model(corrupted_input)
        
        # Loss for backprop (MSE with ground truth)
        loss = F.mse_loss(output, clean)
        
        self.model.zero_grad()
        loss.backward()
        
        lam = torch.abs(corrupted_input.grad)
        lam = lam.mean(dim=1, keepdim=True)
        lam = (lam - lam.min()) / (lam.max() - lam.min() + 1e-8)
        
        return lam.detach()
    
    def visualize_spectral_density(self, image):
        """Compute Fourier Transform for Frequency Analysis 

[Image of Fourier Transform in image processing]
"""
        # image: torch tensor [C, H, W]
        img_np = image.mean(dim=0).cpu().numpy()
        f = np.fft.fft2(img_np)
        fshift = np.fft.fftshift(f)
        magnitude_spectrum = 20 * np.log(np.abs(fshift) + 1e-8)
        return magnitude_spectrum

    def visualize_pixel_distribution(self, ax, image, label, color):
        """Visualize histogram of pixel intensities"""
        img_np = image.cpu().numpy().flatten()
        ax.hist(img_np, bins=50, color=color, alpha=0.5, label=label, density=True)

    def save_individual_components(self, corrupted, restored, clean, image_id, save_dir):
        """Save individual image components and analysis maps"""
        from PIL import Image
        
        # Create subdirectory for individual components
        individual_dir = Path(save_dir) / 'individual_components' / f'{image_id}'
        individual_dir.mkdir(parents=True, exist_ok=True)
        
        to_pil = transforms.ToPILImage()
        
        # 1. Save basic images
        corrupted_img = to_pil(torch.clamp(corrupted, 0.0, 1.0))
        corrupted_img.save(individual_dir / f'{image_id}_01_corrupted.png')
        
        restored_img = to_pil(torch.clamp(restored, 0.0, 1.0))
        restored_img.save(individual_dir / f'{image_id}_02_restored.png')
        
        clean_img = to_pil(torch.clamp(clean, 0.0, 1.0))
        clean_img.save(individual_dir / f'{image_id}_03_ground_truth.png')
        
        # 2. Save difference map
        diff = torch.abs(restored - clean).mean(dim=0)
        diff_np = diff.cpu().numpy()
        diff_colored = plt.cm.hot(diff_np / (diff_np.max() + 1e-8))[:, :, :3]
        diff_colored = (diff_colored * 255).astype(np.uint8)
        diff_img = Image.fromarray(diff_colored)
        diff_img.save(individual_dir / f'{image_id}_04_difference_map.png')
        
        # 3. Save LAM
        with torch.enable_grad():
            lam = self.compute_lam(corrupted.unsqueeze(0), restored.unsqueeze(0), clean.unsqueeze(0))
        
        lam_np = lam.squeeze().cpu().numpy()
        lam_colored = plt.cm.jet(lam_np)[:, :, :3]
        lam_colored = (lam_colored * 255).astype(np.uint8)
        lam_img = Image.fromarray(lam_colored)
        lam_img.save(individual_dir / f'{image_id}_05_lam.png')
        
        # 4. Save LAM overlay
        corrupted_np = torch.clamp(corrupted, 0.0, 1.0).cpu().permute(1, 2, 0).numpy()
        heatmap = plt.cm.jet(lam_np)[:, :, :3]
        overlay = np.clip(0.6 * corrupted_np + 0.4 * heatmap, 0.0, 1.0)
        overlay = (overlay * 255).astype(np.uint8)
        overlay_img = Image.fromarray(overlay)
        overlay_img.save(individual_dir / f'{image_id}_06_lam_overlay.png')
        
        # 5. Save high attribution regions (binary mask)
        lam_binary = (lam_np > lam_np.mean()).astype(np.uint8) * 255
        lam_binary_img = Image.fromarray(lam_binary, mode='L')
        lam_binary_img.save(individual_dir / f'{image_id}_07_high_attribution.png')
        
        # 6. Save edge maps
        sobel_x = torch.tensor([[[[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]]]).float().to(self.device)
        sobel_y = torch.tensor([[[[-1, -2, -1], [0, 0, 0], [1, 2, 1]]]]).float().to(self.device)
        
        restored_gray = restored.mean(dim=0, keepdim=True).unsqueeze(0)
        clean_gray = clean.mean(dim=0, keepdim=True).unsqueeze(0)
        
        restored_edges = torch.sqrt(
            F.conv2d(restored_gray, sobel_x, padding=1)**2 + 
            F.conv2d(restored_gray, sobel_y, padding=1)**2
        )
        clean_edges = torch.sqrt(
            F.conv2d(clean_gray, sobel_x, padding=1)**2 + 
            F.conv2d(clean_gray, sobel_y, padding=1)**2
        )
        
        restored_edges_np = restored_edges.squeeze().cpu().numpy()
        restored_edges_np = (restored_edges_np / (restored_edges_np.max() + 1e-8) * 255).astype(np.uint8)
        restored_edges_img = Image.fromarray(restored_edges_np, mode='L')
        restored_edges_img.save(individual_dir / f'{image_id}_08_restored_edges.png')
        
        clean_edges_np = clean_edges.squeeze().cpu().numpy()
        clean_edges_np = (clean_edges_np / (clean_edges_np.max() + 1e-8) * 255).astype(np.uint8)
        clean_edges_img = Image.fromarray(clean_edges_np, mode='L')
        clean_edges_img.save(individual_dir / f'{image_id}_09_ground_truth_edges.png')
        
        # 7. Save edge difference map
        edge_diff = torch.abs(restored_edges - clean_edges)
        edge_diff_np = edge_diff.squeeze().cpu().numpy()
        edge_diff_colored = plt.cm.hot(edge_diff_np / (edge_diff_np.max() + 1e-8))[:, :, :3]
        edge_diff_colored = (edge_diff_colored * 255).astype(np.uint8)
        edge_diff_img = Image.fromarray(edge_diff_colored)
        edge_diff_img.save(individual_dir / f'{image_id}_10_edge_difference.png')
        
        logger.info(f"Saved individual components to {individual_dir}")
    
    def visualize_lam(self, corrupted, restored, clean, save_path):
        """Visualize LAM"""
        with torch.enable_grad():
            lam = self.compute_lam(corrupted.unsqueeze(0), restored.unsqueeze(0), clean.unsqueeze(0))
        
        corrupted_np = torch.clamp(corrupted, 0.0, 1.0).cpu().permute(1, 2, 0).numpy()
        lam_np = lam.squeeze().cpu().numpy()
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        axes[0].imshow(corrupted_np)
        axes[0].set_title('Corrupted Input', fontsize=16, fontweight='bold')
        axes[0].axis('off')
        
        im1 = axes[1].imshow(lam_np, cmap='jet', interpolation='bilinear')
        axes[1].set_title('Local Attribution Map', fontsize=16, fontweight='bold')
        axes[1].axis('off')
        plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
        
        overlay = corrupted_np.copy()
        heatmap = plt.cm.jet(lam_np)[:, :, :3]
        overlay = np.clip(0.6 * overlay + 0.4 * heatmap, 0.0, 1.0)
        axes[2].imshow(overlay)
        axes[2].set_title('LAM Overlay', fontsize=16, fontweight='bold')
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.close()
    
    def create_comprehensive_analysis(self, corrupted, restored, clean, image_id, save_dir):
        """
        Create comprehensive analysis with LAM, FFT, and Histograms all in one single file.
        This ensures LAM is included in all inference types.
        """
        to_pil = transforms.ToPILImage()
        corrupted_img = to_pil(torch.clamp(corrupted, 0.0, 1.0))
        restored_img = to_pil(torch.clamp(restored, 0.0, 1.0))
        clean_img = to_pil(torch.clamp(clean, 0.0, 1.0))
        
        with torch.no_grad():
            metrics = self.metrics_evaluator.compute(restored.unsqueeze(0), clean.unsqueeze(0))
        
        # Calculate LAM
        with torch.enable_grad():
            lam = self.compute_lam(corrupted.unsqueeze(0), restored.unsqueeze(0), clean.unsqueeze(0))
        lam_np = lam.squeeze().cpu().numpy()

        # Grid Setup: 4 Rows x 4 Columns
        fig = plt.figure(figsize=(24, 20))
        gs = fig.add_gridspec(4, 4, hspace=0.35, wspace=0.35)
        
        # --- Row 1: Visual Comparison ---
        ax1 = fig.add_subplot(gs[0, 0])
        ax1.imshow(corrupted_img)
        ax1.set_title('Corrupted', fontsize=14, fontweight='bold')
        ax1.axis('off')
        
        ax2 = fig.add_subplot(gs[0, 1])
        ax2.imshow(restored_img)
        ax2.set_title('Restored', fontsize=14, fontweight='bold')
        ax2.axis('off')
        
        ax3 = fig.add_subplot(gs[0, 2])
        ax3.imshow(clean_img)
        ax3.set_title('Ground Truth', fontsize=14, fontweight='bold')
        ax3.axis('off')
        
        diff = torch.abs(restored - clean).mean(dim=0)
        ax4 = fig.add_subplot(gs[0, 3])
        im4 = ax4.imshow(diff.cpu().numpy(), cmap='hot', interpolation='bilinear')
        ax4.set_title('Error Heatmap', fontsize=14, fontweight='bold')
        ax4.axis('off')
        plt.colorbar(im4, ax=ax4, fraction=0.046, pad=0.04)
        
        # --- Row 2: LAM (Local Attribution Map) ---
        ax5 = fig.add_subplot(gs[1, 0])
        im5 = ax5.imshow(lam_np, cmap='jet', interpolation='bilinear')
        ax5.set_title('Local Attribution Map (LAM)', fontsize=14, fontweight='bold')
        ax5.axis('off')
        plt.colorbar(im5, ax=ax5, fraction=0.046, pad=0.04)
        
        corrupted_np = torch.clamp(corrupted, 0.0, 1.0).cpu().permute(1, 2, 0).numpy()
        heatmap = plt.cm.jet(lam_np)[:, :, :3]
        overlay = np.clip(0.6 * corrupted_np + 0.4 * heatmap, 0.0, 1.0)
        
        ax6 = fig.add_subplot(gs[1, 1])
        ax6.imshow(overlay)
        ax6.set_title('LAM on Input', fontsize=14, fontweight='bold')
        ax6.axis('off')
        
        # Metrics Display
        ax7 = fig.add_subplot(gs[1, 2:])
        metric_text = '\n'.join([f"{k.upper()}: {v:.4f}" for k, v in metrics.items()])
        ax7.text(0.5, 0.5, metric_text, ha='center', va='center', fontsize=18, 
                 bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8, pad=1.0),
                 family='monospace')
        ax7.set_title('Quality Metrics', fontsize=14, fontweight='bold')
        ax7.axis('off')
        
        # --- Row 3: Frequency Analysis (FFT) ---
        # Frequency spectrum allows us to see how well details (high freq) are restored
        fft_corrupted = self.visualize_spectral_density(corrupted)
        fft_restored = self.visualize_spectral_density(restored)
        fft_clean = self.visualize_spectral_density(clean)
        
        ax8 = fig.add_subplot(gs[2, 0])
        ax8.imshow(fft_corrupted, cmap='viridis')
        ax8.set_title('FFT Corrupted', fontsize=14, fontweight='bold')
        ax8.axis('off')

        ax9 = fig.add_subplot(gs[2, 1])
        ax9.imshow(fft_restored, cmap='viridis')
        ax9.set_title('FFT Restored', fontsize=14, fontweight='bold')
        ax9.axis('off')

        ax10 = fig.add_subplot(gs[2, 2])
        ax10.imshow(fft_clean, cmap='viridis')
        ax10.set_title('FFT Ground Truth', fontsize=14, fontweight='bold')
        ax10.axis('off')
        
        # Histogram
        ax11 = fig.add_subplot(gs[2, 3])
        self.visualize_pixel_distribution(ax11, restored, 'Restored', 'blue')
        self.visualize_pixel_distribution(ax11, clean, 'GT', 'green')
        ax11.set_title('Pixel Histogram', fontsize=14, fontweight='bold')
        ax11.legend()

        # --- Row 4: Edge Maps ---
        sobel_x = torch.tensor([[[[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]]]).float().to(self.device)
        sobel_y = torch.tensor([[[[-1, -2, -1], [0, 0, 0], [1, 2, 1]]]]).float().to(self.device)
        
        restored_gray = restored.mean(dim=0, keepdim=True).unsqueeze(0)
        clean_gray = clean.mean(dim=0, keepdim=True).unsqueeze(0)
        
        restored_edges = torch.sqrt(
            F.conv2d(restored_gray, sobel_x, padding=1)**2 + 
            F.conv2d(restored_gray, sobel_y, padding=1)**2
        )
        clean_edges = torch.sqrt(
            F.conv2d(clean_gray, sobel_x, padding=1)**2 + 
            F.conv2d(clean_gray, sobel_y, padding=1)**2
        )
        
        ax12 = fig.add_subplot(gs[3, 0])
        ax12.imshow(restored_edges.squeeze().cpu().numpy(), cmap='gray')
        ax12.set_title('Restored Edges', fontsize=14, fontweight='bold')
        ax12.axis('off')
        
        ax13 = fig.add_subplot(gs[3, 1])
        ax13.imshow(clean_edges.squeeze().cpu().numpy(), cmap='gray')
        ax13.set_title('Ground Truth Edges', fontsize=14, fontweight='bold')
        ax13.axis('off')
        
        edge_diff = torch.abs(restored_edges - clean_edges)
        ax14 = fig.add_subplot(gs[3, 2])
        im14 = ax14.imshow(edge_diff.squeeze().cpu().numpy(), cmap='hot')
        ax14.set_title('Edge Difference', fontsize=14, fontweight='bold')
        ax14.axis('off')
        
        plt.suptitle(f'MFSSNet Analysis & LAM - {image_id}', fontsize=22, fontweight='bold', y=0.98)
        
        save_path = Path(save_dir) / 'comprehensive' / f'analysis_{image_id}.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.close()
        
        return metrics
    
    def infer_image_set(self, image_paths, save_dir=None):
        """
        Infer a set of images with provided corrupted and ground truth pairs.
        Now includes LAM for each image automatically.
        """
        if save_dir is None:
            save_dir = Path(self.config.analysis_dir) / 'custom_inference'
        else:
            save_dir = Path(save_dir)
        
        save_dir.mkdir(parents=True, exist_ok=True)
        (save_dir / 'comprehensive').mkdir(exist_ok=True)
        (save_dir / 'lam_maps').mkdir(exist_ok=True)
        (save_dir / 'individual_components').mkdir(exist_ok=True)
        
        from PIL import Image
        import torchvision.transforms as transforms
        
        transform = transforms.Compose([
            transforms.ToTensor(),
        ])
        
        all_results = []
        
        # NOTE: We iterate with gradient enabled locally inside visualization functions where needed
        # but keep the main loop no_grad for memory efficiency until LAM calculation.
        with torch.no_grad():
            for idx, (corrupted_path, clean_path) in enumerate(tqdm(image_paths, desc='Processing images')):
                try:
                    # Load images
                    corrupted_img = Image.open(corrupted_path).convert('RGB')
                    clean_img = Image.open(clean_path).convert('RGB')
                    
                    # Convert to tensors
                    corrupted = transform(corrupted_img).unsqueeze(0).to(self.device)
                    clean = transform(clean_img).unsqueeze(0).to(self.device)
                    
                    # Inference
                    restored = torch.clamp(self.model(corrupted), 0.0, 1.0)
                    
                    # Generate image_id from filename
                    corrupted_name = Path(corrupted_path).stem
                    image_id = f'{idx:03d}_{corrupted_name}'
                    
                    # --- COMPREHENSIVE ANALYSIS (Includes LAM, FFT, Histograms) ---
                    # We pass single image tensors (C, H, W)
                    metrics = self.create_comprehensive_analysis(
                        corrupted.squeeze(0), restored.squeeze(0), clean.squeeze(0),
                        image_id, save_dir
                    )
                    
                    # Save standalone LAM visualization
                    lam_save_path = save_dir / 'lam_maps' / f'lam_{image_id}.png'
                    self.visualize_lam(
                        corrupted.squeeze(0), restored.squeeze(0), clean.squeeze(0),
                        lam_save_path
                    )
                    
                    # Save individual components
                    self.save_individual_components(
                        corrupted.squeeze(0), restored.squeeze(0), clean.squeeze(0),
                        image_id, save_dir
                    )
                    
                    # Store results
                    result = {
                        'index': idx,
                        'image_id': image_id,
                        'corrupted_path': str(corrupted_path),
                        'clean_path': str(clean_path),
                        'metrics': metrics,
                        'components_dir': str(save_dir / 'individual_components' / image_id)
                    }
                    all_results.append(result)
                    
                    logger.info(f"Processed {idx+1}/{len(image_paths)}: {image_id}")
                    logger.info(f"  Metrics: " + ", ".join([f"{k}: {v:.4f}" for k, v in metrics.items()]))
                
                except Exception as e:
                    logger.error(f"Error processing {corrupted_path}: {str(e)}")
                    continue
        
        # Summary statistics
        if all_results:
            logger.info("\n" + "="*80)
            logger.info("MFSSNet Custom Image Set Inference Summary")
            logger.info("="*80)
            logger.info(f"Total images processed: {len(all_results)}")
            
            # Aggregate metrics
            all_metrics = defaultdict(list)
            for result in all_results:
                for k, v in result['metrics'].items():
                    all_metrics[k].append(v)
            
            final_metrics = {}
            for k, values in all_metrics.items():
                mean_val = np.mean(values)
                std_val = np.std(values)
                final_metrics[k] = {
                    'mean': mean_val,
                    'std': std_val,
                    'min': np.min(values),
                    'max': np.max(values)
                }
                logger.info(f"{k.upper():>12}: {mean_val:>7.4f} ± {std_val:.4f}")
            logger.info("="*80)
            
            summary = {
                'final_metrics': final_metrics,
                'num_samples': len(all_results),
                'results': all_results,
                'model': 'MFSSNet',
            }
            
            with open(save_dir / 'custom_inference_summary.json', 'w') as f:
                json.dump(summary, f, indent=2, default=str)
        
        return all_results
    
    def batch_inference(self, data_loader, num_samples=100):
        """
        Batch inference using DataLoader.
        Updated to include LAM and comprehensive analysis for every sample processed.
        """
        self.config.create_directories()
        
        output_dir = Path(self.config.analysis_dir)
        all_metrics = defaultdict(list)
        count = 0
        
        with torch.no_grad():
            pbar = tqdm(data_loader, desc='MFSSNet Inference')
            
            for batch in pbar:
                if num_samples is not None and count >= num_samples:
                    break
                
                corrupted = batch['corrupted'].to(self.device)
                clean = batch['clean'].to(self.device)
                image_ids = batch['image_id']
                
                restored = torch.clamp(self.model(corrupted), 0.0, 1.0)
                
                for i in range(min(corrupted.size(0), num_samples - count if num_samples else corrupted.size(0))):
                    
                    # --- LAM & Comprehensive Analysis called here for Batch Inference ---
                    metrics = self.create_comprehensive_analysis(
                        corrupted[i], restored[i], clean[i],
                        image_ids[i], output_dir
                    )
                    
                    # Also save individual LAM file
                    lam_save_path = output_dir / 'lam_maps' / f'lam_{image_ids[i]}.png'
                    self.visualize_lam(corrupted[i], restored[i], clean[i], lam_save_path)
                    
                    # Save individual components
                    self.save_individual_components(
                        corrupted[i], restored[i], clean[i],
                        image_ids[i], output_dir
                    )
                    
                    # Save basic pngs for manipulate dir
                    to_pil = transforms.ToPILImage()
                    corrupted_img = to_pil(torch.clamp(corrupted[i].cpu(), 0.0, 1.0))
                    restored_img = to_pil(torch.clamp(restored[i].cpu(), 0.0, 1.0))
                    clean_img = to_pil(torch.clamp(clean[i].cpu(), 0.0, 1.0))
                    
                    manipulated_dir = Path(self.config.manipulated_dir)
                    corrupted_img.save(manipulated_dir / 'corrupted' / f"{image_ids[i]}.png")
                    restored_img.save(manipulated_dir / 'restored' / f"{image_ids[i]}.png")
                    clean_img.save(manipulated_dir / 'clean' / f"{image_ids[i]}.png")
                    
                    for k, v in metrics.items():
                        all_metrics[k].append(v)
                    
                    count += 1
                
                if all_metrics:
                    current_metrics = {k: sum(v)/len(v) for k, v in all_metrics.items()}
                    pbar.set_postfix({k: f"{v:.3f}" for k, v in current_metrics.items()})
        
        final_metrics = {}
        for k, v in all_metrics.items():
            if v:
                final_metrics[k] = {
                    'mean': sum(v) / len(v),
                    'std': np.std(v),
                    'min': min(v),
                    'max': max(v)
                }
        
        logger.info("=== MFSSNet Evaluation Results ===")
        for k, stats in final_metrics.items():
            logger.info(f"{k.upper()}: {stats['mean']:.4f} ± {stats['std']:.4f}")
        
        summary = {
            'final_metrics': final_metrics,
            'num_samples': count,
            'model': 'MFSSNet',
        }
        
        with open(output_dir / 'evaluation_summary.json', 'w') as f:
            json.dump(summary, f, indent=2, default=str)
        
        return final_metrics


# ============================================================================
# MAIN
# ============================================================================

def main():
    """Main training and evaluation function for MFSSNet"""
    config = Config()
    config.batch_size = 8
    config.epochs = 1500
    config.lr = 1e-4
    config.resume = True
    config.img_size = (64, 256)
    config.metrics = ['psnr', 'ssim']
    
    # Mixed Precision & Optimization
    config.use_amp = True  # Enable Mixed Precision
    config.grad_clip_norm = 0.5  # Gradient Clipping
    
    # Advanced Scheduler (no longer depends on total epochs)
    config.warmup_epochs = 5
    config.scheduler_t0  = 50  # First restart at 50 epochs
    config.scheduler_tmult = 2  # Double the period after each restart
    
    config.create_directories()
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    print(f"Model: MFSSNet with CA-SSM + CG-CSWA")
    print(f"Mixed Precision: {'ENABLED' if config.use_amp else 'DISABLED'}")
    print(f"Gradient Clipping: {config.grad_clip_norm}")
    print(f"Scheduler: Warmup({config.warmup_epochs}E) + CosineAnnealingWarmRestarts(T0={config.scheduler_t0}, Tmult={config.scheduler_tmult}) + ReduceLROnPlateau")
    
    # Datasets
    train_dataset = MergedTextDataset(
        synthetic_dir='/kaggle/working/stt_train',
        handwritten_dir='/kaggle/working/htt_train',
        split='train',
        img_size=config.img_size,
        augment=True,
        config=config
    )
    
    val_dataset = MergedTextDataset(
        synthetic_dir='/kaggle/working/stt_test',
        handwritten_dir='/kaggle/working/htt_test',
        split='val',
        img_size=config.img_size,
        augment=False,
        config=config
    )
    
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, num_workers=4, pin_memory=True)
    
    print(f"Train: {len(train_dataset)} samples, Val: {len(val_dataset)} samples")
    
    # Initialize MFSSNet Model with Novel Components
    model = MFSSNet(
        in_channels=config.in_channels,
        out_channels=config.out_channels,
        dim=config.dim,
        num_blocks=config.num_blocks,
        num_heads=config.num_heads,
        window_sizes=config.window_sizes,
        patch_size=config.patch_size,
        num_layers=config.num_prt_layers
    )
    print(f"MFSSNet initialized with {sum(p.numel() for p in model.parameters()):,} parameters")
    
    # Trainer
    trainer = MFSSNetTrainer(model, train_loader, val_loader, device, config)
    
    if config.resume:
        trainer.load_checkpoint()
    
    # Train
    trainer.train(config.epochs)
    
    # Inference
    print("Starting MFSSNet inference...")
    inferencer = MFSSNetInferencer(model, device, config, checkpoint_path=trainer.checkpoint_dir / 'best.pth')
    
    final_metrics = inferencer.batch_inference(val_loader, num_samples=600)

    print(final_metrics)

    # Prepare your image pairs
    image_pairs = [
        ('/kaggle/working/stt_test/t_incomplete/0000.png', '/kaggle/working/stt_test/t_f/0000.png'),
        ('/kaggle/working/stt_test/t_incomplete/0004.png', '/kaggle/working/stt_test/t_f/0004.png'),
        ('/kaggle/working/stt_test/t_incomplete/0005.png', '/kaggle/working/stt_test/t_f/0005.png'),
        ('/kaggle/working/stt_test/t_incomplete/0006.png', '/kaggle/working/stt_test/t_f/0006.png'),
        ('/kaggle/working/stt_test/t_incomplete/0007.png', '/kaggle/working/stt_test/t_f/0007.png'),
        ('/kaggle/working/stt_test/t_incomplete/0008.png', '/kaggle/working/stt_test/t_f/0008.png'),
        ('/kaggle/working/stt_test/t_incomplete/0009.png', '/kaggle/working/stt_test/t_f/0009.png'),
        ('/kaggle/working/stt_test/t_incomplete/0010.png', '/kaggle/working/stt_test/t_f/0010.png'),
        ('/kaggle/working/stt_test/t_incomplete/0011.png', '/kaggle/working/stt_test/t_f/0011.png'),
        ('/kaggle/working/stt_test/t_incomplete/0012.png', '/kaggle/working/stt_test/t_f/0012.png'),
        ('/kaggle/working/stt_test/t_incomplete/0013.png', '/kaggle/working/stt_test/t_f/0013.png'),
        ('/kaggle/working/stt_test/t_incomplete/0014.png', '/kaggle/working/stt_test/t_f/0014.png'),
        ('/kaggle/working/stt_test/t_incomplete/0015.png', '/kaggle/working/stt_test/t_f/0015.png'),
        ('/kaggle/working/stt_test/t_incomplete/0016.png', '/kaggle/working/stt_test/t_f/0016.png'),
        ('/kaggle/working/stt_test/t_incomplete/0017.png', '/kaggle/working/stt_test/t_f/0017.png'),
        ('/kaggle/working/stt_test/t_incomplete/0018.png', '/kaggle/working/stt_test/t_f/0018.png'),
        ('/kaggle/working/stt_test/t_incomplete/0019.png', '/kaggle/working/stt_test/t_f/0019.png'),
        ('/kaggle/working/stt_test/t_incomplete/0020.png', '/kaggle/working/stt_test/t_f/0020.png'),
        ('/kaggle/working/stt_test/t_incomplete/0021.png', '/kaggle/working/stt_test/t_f/0021.png'),
        ('/kaggle/working/stt_test/t_incomplete/0022.png', '/kaggle/working/stt_test/t_f/0022.png'),
        ('/kaggle/working/stt_test/t_incomplete/0023.png', '/kaggle/working/stt_test/t_f/0023.png'),
        ('/kaggle/working/stt_test/t_incomplete/0024.png', '/kaggle/working/stt_test/t_f/0024.png'),
        ('/kaggle/working/stt_test/t_incomplete/0025.png', '/kaggle/working/stt_test/t_f/0025.png'),
        ('/kaggle/working/stt_test/t_incomplete/0026.png', '/kaggle/working/stt_test/t_f/0026.png'),
        ('/kaggle/working/stt_test/t_incomplete/0027.png', '/kaggle/working/stt_test/t_f/0027.png'),
        ('/kaggle/working/stt_test/t_incomplete/0062.png', '/kaggle/working/stt_test/t_f/0062.png'),
        ('/kaggle/working/stt_test/t_incomplete/0063.png', '/kaggle/working/stt_test/t_f/0063.png'),
        ('/kaggle/working/stt_test/t_incomplete/0064.png', '/kaggle/working/stt_test/t_f/0064.png'),
        ('/kaggle/working/stt_test/t_incomplete/0065.png', '/kaggle/working/stt_test/t_f/0065.png'),
        ('/kaggle/working/stt_test/t_incomplete/0066.png', '/kaggle/working/stt_test/t_f/0066.png'),
        ('/kaggle/working/stt_test/t_incomplete/0067.png', '/kaggle/working/stt_test/t_f/0067.png'),
        ('/kaggle/working/stt_test/t_incomplete/0068.png', '/kaggle/working/stt_test/t_f/0068.png'),
        ('/kaggle/working/stt_test/t_incomplete/0069.png', '/kaggle/working/stt_test/t_f/0069.png'),
        ('/kaggle/working/stt_test/t_incomplete/0070.png', '/kaggle/working/stt_test/t_f/0070.png'),
        ('/kaggle/working/stt_test/t_incomplete/0071.png', '/kaggle/working/stt_test/t_f/0071.png'),
        ('/kaggle/working/stt_test/t_incomplete/0072.png', '/kaggle/working/stt_test/t_f/0072.png'),
        ('/kaggle/working/stt_test/t_incomplete/0073.png', '/kaggle/working/stt_test/t_f/0073.png'),
        ('/kaggle/working/stt_test/t_incomplete/0074.png', '/kaggle/working/stt_test/t_f/0074.png'),
        ('/kaggle/working/stt_test/t_incomplete/0075.png', '/kaggle/working/stt_test/t_f/0075.png'),
        ('/kaggle/working/stt_test/t_incomplete/0076.png', '/kaggle/working/stt_test/t_f/0076.png'),
        ('/kaggle/working/stt_test/t_incomplete/0077.png', '/kaggle/working/stt_test/t_f/0077.png'),
        ('/kaggle/working/stt_test/t_incomplete/0078.png', '/kaggle/working/stt_test/t_f/0078.png'),
        ('/kaggle/working/stt_test/t_incomplete/0079.png', '/kaggle/working/stt_test/t_f/0079.png'),
        ('/kaggle/working/stt_test/t_incomplete/0080.png', '/kaggle/working/stt_test/t_f/0080.png'),
        ('/kaggle/working/stt_test/t_incomplete/0081.png', '/kaggle/working/stt_test/t_f/0081.png'),
        ('/kaggle/working/stt_test/t_incomplete/0082.png', '/kaggle/working/stt_test/t_f/0082.png'),
        ('/kaggle/working/stt_test/t_incomplete/0083.png', '/kaggle/working/stt_test/t_f/0083.png'),
        ('/kaggle/working/stt_test/t_incomplete/0084.png', '/kaggle/working/stt_test/t_f/0084.png'),
        ('/kaggle/working/stt_test/t_incomplete/0085.png', '/kaggle/working/stt_test/t_f/0085.png'),
        ('/kaggle/working/stt_test/t_incomplete/0086.png', '/kaggle/working/stt_test/t_f/0086.png'),
        ('/kaggle/working/stt_test/t_incomplete/0087.png', '/kaggle/working/stt_test/t_f/0087.png'),
        ('/kaggle/working/stt_test/t_incomplete/0088.png', '/kaggle/working/stt_test/t_f/0088.png'),
        ('/kaggle/working/stt_test/t_incomplete/0089.png', '/kaggle/working/stt_test/t_f/0089.png'),
        ('/kaggle/working/stt_test/t_incomplete/0090.png', '/kaggle/working/stt_test/t_f/0090.png'),
        ('/kaggle/working/stt_test/t_incomplete/0091.png', '/kaggle/working/stt_test/t_f/0091.png'),
        ('/kaggle/working/stt_test/t_incomplete/0092.png', '/kaggle/working/stt_test/t_f/0092.png'),
        ('/kaggle/working/stt_test/t_incomplete/0093.png', '/kaggle/working/stt_test/t_f/0093.png'),
        ('/kaggle/working/stt_test/t_incomplete/0094.png', '/kaggle/working/stt_test/t_f/0094.png'),
        ('/kaggle/working/stt_test/t_incomplete/0095.png', '/kaggle/working/stt_test/t_f/0095.png'),
        ('/kaggle/working/stt_test/t_incomplete/0096.png', '/kaggle/working/stt_test/t_f/0096.png'),
        ('/kaggle/working/stt_test/t_incomplete/0097.png', '/kaggle/working/stt_test/t_f/0097.png'),
        ('/kaggle/working/stt_test/t_incomplete/0098.png', '/kaggle/working/stt_test/t_f/0098.png'),
        ('/kaggle/working/stt_test/t_incomplete/0099.png', '/kaggle/working/stt_test/t_f/0099.png'),
        ('/kaggle/working/stt_test/t_incomplete/0100.png', '/kaggle/working/stt_test/t_f/0100.png'),
        ('/kaggle/working/stt_test/t_incomplete/0101.png', '/kaggle/working/stt_test/t_f/0101.png'),
        ('/kaggle/working/stt_test/t_incomplete/0102.png', '/kaggle/working/stt_test/t_f/0102.png'),
        ('/kaggle/working/stt_test/t_incomplete/0103.png', '/kaggle/working/stt_test/t_f/0103.png'),
        ('/kaggle/working/stt_test/t_incomplete/0104.png', '/kaggle/working/stt_test/t_f/0104.png'),
        ('/kaggle/working/stt_test/t_incomplete/0105.png', '/kaggle/working/stt_test/t_f/0105.png'),
        ('/kaggle/working/stt_test/t_incomplete/0106.png', '/kaggle/working/stt_test/t_f/0106.png'),
        ('/kaggle/working/stt_test/t_incomplete/0107.png', '/kaggle/working/stt_test/t_f/0107.png'),
        ('/kaggle/working/stt_test/t_incomplete/0367.png', '/kaggle/working/stt_test/t_f/0367.png'),
        ('/kaggle/working/stt_test/t_incomplete/0537.png', '/kaggle/working/stt_test/t_f/0537.png'),
        ('/kaggle/working/stt_test/t_incomplete/0564.png', '/kaggle/working/stt_test/t_f/0564.png'),
        ('/kaggle/working/stt_test/t_incomplete/0565.png', '/kaggle/working/stt_test/t_f/0565.png'),
        ('/kaggle/working/stt_test/t_incomplete/0599.png', '/kaggle/working/stt_test/t_f/0599.png'),
        ('/kaggle/working/stt_test/t_incomplete/0733.png', '/kaggle/working/stt_test/t_f/0733.png'),
        ('/kaggle/working/stt_test/t_incomplete/0734.png', '/kaggle/working/stt_test/t_f/0734.png'),
    ]

    image_pairs = [
        ('/kaggle/working/htt_test/Corrupted_Image/00000.png', '/kaggle/working/htt_test/Intact_Image/00000.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00001.png', '/kaggle/working/htt_test/Intact_Image/00001.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00002.png', '/kaggle/working/htt_test/Intact_Image/00002.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00003.png', '/kaggle/working/htt_test/Intact_Image/00003.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00004.png', '/kaggle/working/htt_test/Intact_Image/00004.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00005.png', '/kaggle/working/htt_test/Intact_Image/00005.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00006.png', '/kaggle/working/htt_test/Intact_Image/00006.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00007.png', '/kaggle/working/htt_test/Intact_Image/00007.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00008.png', '/kaggle/working/htt_test/Intact_Image/00008.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00009.png', '/kaggle/working/htt_test/Intact_Image/00009.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00010.png', '/kaggle/working/htt_test/Intact_Image/00010.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00011.png', '/kaggle/working/htt_test/Intact_Image/00011.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00012.png', '/kaggle/working/htt_test/Intact_Image/00012.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00013.png', '/kaggle/working/htt_test/Intact_Image/00013.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00014.png', '/kaggle/working/htt_test/Intact_Image/00014.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00015.png', '/kaggle/working/htt_test/Intact_Image/00015.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00016.png', '/kaggle/working/htt_test/Intact_Image/00016.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00017.png', '/kaggle/working/htt_test/Intact_Image/00017.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00018.png', '/kaggle/working/htt_test/Intact_Image/00018.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00019.png', '/kaggle/working/htt_test/Intact_Image/00019.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00020.png', '/kaggle/working/htt_test/Intact_Image/00020.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00021.png', '/kaggle/working/htt_test/Intact_Image/00021.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00022.png', '/kaggle/working/htt_test/Intact_Image/00022.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00023.png', '/kaggle/working/htt_test/Intact_Image/00023.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00024.png', '/kaggle/working/htt_test/Intact_Image/00024.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/00025.png', '/kaggle/working/htt_test/Intact_Image/00025.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/01000.png', '/kaggle/working/htt_test/Intact_Image/01000.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/01001.png', '/kaggle/working/htt_test/Intact_Image/01001.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/01002.png', '/kaggle/working/htt_test/Intact_Image/01002.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/01003.png', '/kaggle/working/htt_test/Intact_Image/01003.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/01004.png', '/kaggle/working/htt_test/Intact_Image/01004.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/01005.png', '/kaggle/working/htt_test/Intact_Image/01005.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/01006.png', '/kaggle/working/htt_test/Intact_Image/01006.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/01007.png', '/kaggle/working/htt_test/Intact_Image/01007.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/01008.png', '/kaggle/working/htt_test/Intact_Image/01008.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/01009.png', '/kaggle/working/htt_test/Intact_Image/01009.png'),
    ]

    image_pairs = [
        ('/kaggle/working/htt_test/Corrupted_Image/01509.png', '/kaggle/working/htt_test/Intact_Image/01509.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/01510.png', '/kaggle/working/htt_test/Intact_Image/01510.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/01511.png', '/kaggle/working/htt_test/Intact_Image/01511.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/01512.png', '/kaggle/working/htt_test/Intact_Image/01512.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/01513.png', '/kaggle/working/htt_test/Intact_Image/01513.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/01514.png', '/kaggle/working/htt_test/Intact_Image/01514.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/01515.png', '/kaggle/working/htt_test/Intact_Image/01515.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/01516.png', '/kaggle/working/htt_test/Intact_Image/01516.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/01517.png', '/kaggle/working/htt_test/Intact_Image/01517.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/01518.png', '/kaggle/working/htt_test/Intact_Image/01518.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/01519.png', '/kaggle/working/htt_test/Intact_Image/01519.png'),
        ('/kaggle/working/htt_test/Corrupted_Image/01520.png', '/kaggle/working/htt_test/Intact_Image/01520.png'),
    ]
    
    # Run inference
    results = inferencer.infer_image_set(
        image_paths=image_pairs,
        save_dir='htt_results'  # Optional
    )
    
    # Access results
    for result in results:
        print(f"Image {result['index']}: PSNR = {result['metrics']['psnr']:.2f}")


if __name__ == '__main__':
    main()

Device: cuda
Model: HIRNet with CA-SSM + CG-CSWA
Mixed Precision: ENABLED
Gradient Clipping: 0.5
Scheduler: Warmup(5E) + CosineAnnealingWarmRestarts(T0=50, Tmult=2) + ReduceLROnPlateau
Train: 5429 samples, Val: 2383 samples
HIRNet initialized with 30,520,729 parameters


2025-12-08 09:15:48,189 INFO: Network [PSNR] is created.
2025-12-08 09:15:48,191 INFO: Metric [PSNR] is created.
2025-12-08 09:15:48,195 INFO: Network [SSIM] is created.
2025-12-08 09:15:48,195 INFO: Metric [SSIM] is created.
/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to